# Hyperparameter tuning

**Run order:** `tuning.ipynb` → `imputation.ipynb` → `downstream.ipynb`

Searches for the best hyperparameters for each method that exposes them (kNN, SoftImpute, DiffPuter, GRAPE) and writes the selected configurations to `results/tuned_params.json`, keyed by dataset name. Mean, MICE, and HyperImpute have no exposed knobs and are recorded as no-ops.

**To switch datasets:** change `DATASET` in Cell 1 to `"scenario5"` or `"scenario33"` and re-run the entire notebook. Each dataset gets its own block in `tuned_params.json`, so running for both datasets is safe and additive.

## 1. Path setup + dataset

In [1]:
# ---------------------------------------------------------------------------
# Path setup + dataset switch. This is the ONLY cell you change between
# datasets: set DATASET to a key registered in common.datasets.
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

DATASET = "scenario33"          # "scenario5" | "scenario33"


def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "src").is_dir() and (parent / "data").is_dir():
            return parent
    return start.parent


REPO_ROOT   = _find_repo_root(Path.cwd())
SRC         = REPO_ROOT / "src"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
CSV_DIR     = RESULTS_DIR / "csv"
TUNED_PARAMS_PATH = RESULTS_DIR / "tuned_params.json"
for _d in (FIGURES_DIR, CSV_DIR):
    _d.mkdir(parents=True, exist_ok=True)

_diffputer = REPO_ROOT / "external" / "DiffPuter"
for _p in (REPO_ROOT, SRC,
           _diffputer / "baselines" / "GRAPE",
           _diffputer / "baselines",
           _diffputer):
    _sp = str(_p.resolve())
    if _p.exists() and _sp not in sys.path:
        sys.path.insert(0, _sp)

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from imputers.imputers import KNNImputerWrapper, SoftImputeWrapper
from imputers.diffputer_imputer import DiffPuterImputer
from imputers.grape_imputer import GRAPEImputer

from common.experiment import ExperimentConfig
from common.tuning import (
    tune_knn, tune_softimpute, softimpute_shrinkage_grid,
    save_tuned_params, load_tuned_params,
)
from common.tuning_optuna import tune_imputer_optuna

from common.datasets import get_dataset

spec = get_dataset(DATASET)
DATA_DIR = REPO_ROOT / "data" / spec.data_dirname
clean_data = spec.loader(DATA_DIR)
print(f"Dataset: {spec.name}  |  clean_data {tuple(clean_data.shape)}")
print("Tuned params ->", TUNED_PARAMS_PATH)

Parsing location files...
Parsing mmWave power files...
Argmax matches unit1_beam: 100.00%
Clean data shape: (3981, 70)
Missing values after parsing: 1270
  (rows with parse failures — consider dropping before experiments)
Dataset: scenario33  |  clean_data (3981, 70)
Tuned params -> /mnt/c/Users/Kenneth Chan/Documents/School/Research Project/Repo/ImputationFor6GDatasets/results/tuned_params.json


## 2. Tuning configuration

`cfg` is built entirely from the spec, so the same cell tunes any dataset.

In [ ]:
cfg = ExperimentConfig(
    target_cols=spec.target_cols,
    retained_cols=spec.retained_cols,
    mar_driver_cols=spec.mar_driver_cols,
    mar_score_mode=spec.mar_score_mode,
    proportions=[0.10, 0.30, 0.50],
    n_seeds=1,
)

TUNING_PROP  = 0.30
TUNING_SEEDS = (0, 1, 2)        # cheap methods can afford 3 seeds
DEEP_SEEDS   = (0,)             # deep methods: single seed for cost

## 3. Cheap methods — grid search

**kNN:** sweeps `n_neighbors` ∈ {3, 5, 10, 20, 50} at 30 % missingness (MCAR), averaged over three seeds. The value minimising mean RMSE is selected.

**SoftImpute:** sweeps `shrinkage_value` (λ) over a data-derived grid (logarithmically spaced between the smallest and largest singular values of the scaled data). A `None` result means the library's automatic threshold won.

In [ ]:
# KNN: n_neighbors
knn_result = tune_knn(
    clean_data, cfg, KNNImputerWrapper,
    grid=(3, 5, 10, 20, 50),
    prop=TUNING_PROP, tuning_seeds=TUNING_SEEDS, verbose=True,
)
display(knn_result["table"])
KNN_K = int(knn_result["best_value"])
print("Selected knn_n_neighbors =", KNN_K)

In [ ]:
# SoftImpute: shrinkage_value (lambda) over a data-derived grid
grid = softimpute_shrinkage_grid(clean_data, cfg)
soft_result = tune_softimpute(
    clean_data, cfg, SoftImputeWrapper,
    grid=grid, prop=TUNING_PROP, tuning_seeds=TUNING_SEEDS, verbose=True,
)
display(soft_result["table"])

_soft_best = soft_result["best_value"]
SOFTIMPUTE_SHRINKAGE = None if (_soft_best is None or pd.isna(_soft_best)) else float(_soft_best)
print("Selected softimpute_shrinkage =", SOFTIMPUTE_SHRINKAGE)

## 4. DiffPuter sensitivity scan

A one-at-a-time scan across five hyperparameters to identify which ones actually move RMSE beyond the noise floor. Parameters whose swing is below the median seed-to-seed noise are fixed at sensible defaults; only the high-swing ones (`lr`, `hid_dim`) enter the joint Optuna search in §5. This two-stage approach keeps the expensive joint search tractable.

In [ ]:
from common.tuning_optuna import _build_masked_sets, _mean_rmse

# Converged per-iteration budget: this is what makes the ranking trustworthy.
# If runtime is tight, trim the NUMBER of values per knob below, not the epochs.
def diffputer_factory(hid_dim=256, n_em_iterations=3, n_train_epochs=5000,
                      n_sampling_trials=10, n_diffusion_steps=50,
                      lr=1e-4, batch_size=1024):
    return DiffPuterImputer(
        hid_dim=hid_dim, n_em_iterations=n_em_iterations,
        n_train_epochs=n_train_epochs, n_sampling_trials=n_sampling_trials,
        n_diffusion_steps=n_diffusion_steps, lr=lr, batch_size=batch_size,
        gaussianize=True, early_stopping_patience=300, verbose=False,
    )

SCAN_GRID = {
    "lr":                [5e-5, 1e-4, 2e-4],
    "hid_dim":           [128, 256, 512],
    "n_diffusion_steps": [50, 100, 200],
    "n_em_iterations":   [1, 3, 6],
    "n_sampling_trials": [5, 10],
}

masked_sets = _build_masked_sets(cfg, clean_data, TUNING_PROP, DEEP_SEEDS)

scan_rows = []
for pname, values in SCAN_GRID.items():
    print(f"\n[{pname}]")
    for v in values:
        rmse_mean, rmse_std = _mean_rmse({pname: v}, diffputer_factory,
                                         masked_sets, DEEP_SEEDS)
        scan_rows.append({"param": pname, "value": v,
                          "rmse_mean": rmse_mean, "rmse_std": rmse_std})
        print(f"  {pname}={v:<8} RMSE {rmse_mean:.4f} ± {rmse_std:.4f}")
scan_df = pd.DataFrame(scan_rows)

swing = (scan_df.groupby("param")["rmse_mean"]
         .agg(lambda s: s.max() - s.min())
         .sort_values(ascending=False))
noise = float(scan_df["rmse_std"].median())
print(f"\nRMSE swing per parameter (noise band ~ {noise:.4f}):")
print(swing.to_string())

In [ ]:
# Hold lr/hid_dim near the converged optimum; sweep EM only.
EM_PROBE = {"lr": 1e-4, "hid_dim": 512,
            "n_train_epochs": 5000, "n_sampling_trials": 10}

em_rows = []
for em in [1, 2, 3, 4, 5, 6]:
    rmse_mean, rmse_std = _mean_rmse({**EM_PROBE, "n_em_iterations": em},
                                     diffputer_factory, masked_sets, DEEP_SEEDS)
    em_rows.append({"n_em_iterations": em,
                    "rmse_mean": rmse_mean, "rmse_std": rmse_std})
    print(f"  EM={em:<2} RMSE {rmse_mean:.4f} ± {rmse_std:.4f}")
em_df = pd.DataFrame(em_rows)

# smallest EM within `tol` of the best observed RMSE
tol = max(noise, 5e-3)
best_rmse = em_df["rmse_mean"].min()
EM_PLATEAU = int(em_df.loc[em_df["rmse_mean"] <= best_rmse + tol,
                           "n_em_iterations"].min())
print(f"\nChosen n_em_iterations (plateau within {tol:.4f}): {EM_PLATEAU}")

## 5. DiffPuter — joint Optuna search

Joint TPE search over `lr` (log-uniform) and `hid_dim` (categorical), with `n_em_iterations`, epochs, and sampling trials fixed at the plateau found in §4. A small complexity penalty discourages larger `hid_dim` when the RMSE difference is negligible. 15 trials with a fixed sampler seed for reproducibility.

In [ ]:
FIXED_EM_ITERS = 3
FIXED_EPOCHS   = 5000
FIXED_SAMPLING = 10
FIXED_PATIENCE = 300
FIXED_DIFFUSION_STEPS = 50
def diffputer_search_factory(**tuned):
    return DiffPuterImputer(
        gaussianize=True,
        n_em_iterations=FIXED_EM_ITERS,
        n_train_epochs=FIXED_EPOCHS,
        n_sampling_trials=FIXED_SAMPLING,
        early_stopping_patience=FIXED_PATIENCE,
        n_diffusion_steps=FIXED_DIFFUSION_STEPS,
        verbose=False,
        **tuned,
    )

SEARCH_SPACE = {
    "lr":      ("loguniform", 5e-5, 2e-4),
    "hid_dim": ("categorical", [128, 256, 512]),
}

diffputer_opt = tune_imputer_optuna(
    clean_data, cfg,
    factory=diffputer_search_factory,
    search_space=SEARCH_SPACE,
    prop=TUNING_PROP,
    tuning_seeds=DEEP_SEEDS,
    n_trials=15,
    sampler_seed=0,
    complexity_key="hid_dim",
    complexity_weight=1e-3,
    verbose=True,
)
display(diffputer_opt["table"])
DIFFPUTER_PARAMS = dict(diffputer_opt["best_params"])
print(f"Selected DiffPuter params: {DIFFPUTER_PARAMS} "
      f"(RMSE {diffputer_opt['best_rmse']:.4f})")

## 6. GRAPE — joint Optuna search
Joint TPE search over training budget (`epochs`), model capacity (`node_edge_dim` applied to both node and edge MLPs), and learning rate. The same `node_edge_dim` is used for both node and edge dimensions to keep the search space tractable. 20 trials at 30 % MCAR with a single seed (GRAPE is too slow for multi-seed tuning).

In [ ]:
grape_opt = tune_imputer_optuna(
    clean_data, cfg,
    factory=lambda epochs, node_edge_dim, lr: GRAPEImputer(
        epochs=epochs, node_dim=node_edge_dim, edge_dim=node_edge_dim, lr=lr,
    ),
    search_space={
        "epochs":        ("categorical", [2000, 5000, 10000, 20000]),
        "node_edge_dim": ("categorical", [32, 64, 128]),
        "lr":            ("loguniform", 3e-4, 3e-3),
    },
    prop=TUNING_PROP, tuning_seeds=DEEP_SEEDS,
    n_trials=20,
    sampler_seed=0,
    complexity_key="node_edge_dim", complexity_weight=1e-3,
    verbose=True,
)
display(grape_opt["table"])
GRAPE_EPOCHS   = int(grape_opt["best_params"]["epochs"])
GRAPE_CAPACITY = int(grape_opt["best_params"]["node_edge_dim"])
GRAPE_LR       = float(grape_opt["best_params"]["lr"])
print(f"Selected GRAPE epochs={GRAPE_EPOCHS}, node/edge dim={GRAPE_CAPACITY}, "
      f"lr={GRAPE_LR:.2e} (RMSE {grape_opt['best_rmse']:.4f})")

## 7. Save tuned parameters

Writes all selected values to `results/tuned_params.json` under a key equal to `spec.name` (e.g. `"scenario5"`). Running this cell for a second dataset merges into the same file without overwriting the first dataset's block. `imputation.ipynb` and `downstream.ipynb` both load from this file and will assert-fail loudly if a required key is missing.

In [ ]:
diffputer_full_config = {
    **DIFFPUTER_PARAMS,
    "gaussianize": True,
    "n_em_iterations": FIXED_EM_ITERS,
    "n_train_epochs": FIXED_EPOCHS,
    "n_sampling_trials": FIXED_SAMPLING,
    "early_stopping_patience": FIXED_PATIENCE,
}

save_tuned_params(TUNED_PARAMS_PATH, spec.name, {
    "knn_n_neighbors":      int(KNN_K),
    "softimpute_shrinkage": SOFTIMPUTE_SHRINKAGE,
    "diffputer":            diffputer_full_config,
    "grape_epochs":         int(GRAPE_EPOCHS),
    "grape_node_edge_dim":  int(GRAPE_CAPACITY),
    "grape_lr":             float(GRAPE_LR),
})
print("Saved for", spec.name, "->", load_tuned_params(TUNED_PARAMS_PATH, spec.name))